# Neural networks on the text track (Phase A1 of `reports/05_implementation_plan.md`)

Feed-forward networks on the tweeted stock-days of `text_master.pkl`, with the same walk-forward
design as `03a/prediction_linear_regression_text_only.ipynb` (monthly refit, 252-trading-day
window, daily out-of-sample predictions, `index = mm_index`) and the Gu-Kelly-Xiu (2020) training
recipe: ReLU, batch normalisation, Adam, early stopping on a temporal validation block, a weight
penalty chosen on that block, and an ensemble of random seeds averaged.

| switch (env var) | values | meaning |
|---|---|---|
| `NN_ARCH` (`TEXTNN_ARCH`) | `1`, `2`, `3` | hidden layers 64 / 64-32 / 128-64-32 |
| `FEATURE_SET` (`TEXTONLY_FEATURES`) | `core`, `all`, `embed+norm+core`, `embed+norm+all` | 2 / 53 / 388 / 439 regressors, same names and output tags as the linear track (`core`, `all`, `textcore`, `textall`) |
| `RANK_TARGET` | `0/1` | train on the daily percentile rank of the target minus 0.5 (default 1) |
| `TARGET_COL` | `f_cumret1`, `ar_dgtw_1` | raw or DGTW-adjusted next-day return (tag `_dgtw`) |
| `RANK_FEATURES` (`TEXTNN_RANK_FEATURES`) | `0/1` | daily rank of every non-embedding feature to [-1, 1] (GKX); embedding columns untouched (default 1; tag `_norank` when off) |
| `PLACEBO` (`TEXTNN_PLACEBO`) | integer | append that many standard-normal noise columns (tag `_placebo`); their importance is a check on overfitting |
| `N_SEEDS`, `N_WORKERS`, `MAX_EPOCHS`, `MAX_MONTHS` (`TEXTNN_*`) | integers | ensemble size, parallel worker processes, epoch cap, and a month cap for smoke tests (0 = all) |

**Per month.** The 252-day window is split by date: the first 80% of its dates train, the last 20%
validate. For every weight decay in `WD_GRID` and every seed, a network is trained on the training
block with early stopping on the validation block's mean daily rank correlation (patience
`PATIENCE`); the weight decay with the best mean validation correlation across seeds is chosen,
and the month's prediction is the average of that setting's seed networks. The realised
out-of-sample rank correlation of *every* candidate is stored in the sidecar, so the linear
track's walk-forward selection rule can be applied to the same candidates afterwards.

**Diagnostics** (validation block, chosen setting): group importance (drop in rank correlation
when a feature group is set to its training mean), placebo columns, and the marginal response of
the prediction to `log_volume` and `net_sentiment` (others at their median) plus their
interaction surface.

Output: a configuration-specific directory under `Code/.runs/text_nn/` (override with
`TEXTNN_RUN_ROOT`). Each month is atomically checkpointed and reused after interruption.
Run identity includes the input file size/mtime, ordered feature names, code hashes,
library versions, hyperparameters, seeds and requested months. Changed configurations
produce separate directories. Pilots never replace full-run or existing prediction files.

For a pilot, set `TEXTNN_MONTHS=2012-01,2022-12`, `TEXTNN_SEEDS=1`,
`TEXTNN_MAX_EPOCHS=3`, `TEXTNN_WORKERS=2`, `TEXTNN_THREADS=2`. These truncated fits test
the execution path and runtime; they are not estimates of the fully tuned model's quality.
Re-run the same command to verify resume. Final publication is atomic, with a
`complete.json` marker written only after predictions and metadata are saved.

Feature availability determines the prediction universe. Training and validation
use observed labels, while missing future labels do not remove prediction rows or change
the daily ranks of the inputs. All rank-correlation scoring uses the shared evaluator rule.


In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import sys
import hashlib
import importlib.metadata
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
INPUT_DATA = MODEL_DATA_DIR / "text_master.pkl"
RUN_ROOT = Path(os.environ.get("TEXTNN_RUN_ROOT", str(CODE_DIR / ".runs" / "text_nn")))
sys.path.insert(0, str(CODE_DIR / "tools"))
from nn_checkpoint import fingerprint, atomic_pickle, atomic_json, fit_month_checkpoint
from filelock import FileLock

# =============================================================================
# SWITCHES (environment variables override for headless runs)
# =============================================================================
NN_ARCH       = int(os.environ.get("TEXTNN_ARCH", "3"))                    # 1 | 2 | 3
FEATURE_SET   = os.environ.get("TEXTONLY_FEATURES", "embed+norm+core")     # core | all | embed+norm+core | embed+norm+all
RANK_TARGET   = os.environ.get("RANK_TARGET", "1") == "1"
TARGET_COL    = os.environ.get("TARGET_COL", "f_cumret1")
RANK_FEATURES = os.environ.get("TEXTNN_RANK_FEATURES", "1") == "1"        # daily rank of non-embedding features to [-1, 1]
PLACEBO       = int(os.environ.get("TEXTNN_PLACEBO", "0"))                 # number of noise columns appended
N_SEEDS       = int(os.environ.get("TEXTNN_SEEDS", "5"))
N_WORKERS     = int(os.environ.get("TEXTNN_WORKERS", "8"))                 # parallel processes (months)
THREADS_PER_WORKER = int(os.environ.get("TEXTNN_THREADS", "4"))
MAX_EPOCHS    = int(os.environ.get("TEXTNN_MAX_EPOCHS", "100"))
MAX_MONTHS    = int(os.environ.get("TEXTNN_MAX_MONTHS", "0"))              # 0 = all OOS months; >0 for smoke tests

RUN_MONTHS = [m.strip() for m in os.environ.get("TEXTNN_MONTHS", "").split(",") if m.strip()]
assert N_SEEDS > 0 and N_WORKERS > 0 and THREADS_PER_WORKER > 0 and MAX_EPOCHS > 0 and MAX_MONTHS >= 0
IS_PILOT = bool(MAX_MONTHS or RUN_MONTHS or N_SEEDS < 5 or MAX_EPOCHS < 100)
RUN_KIND = "pilot" if IS_PILOT else "full"

ARCHS = {1: [64], 2: [64, 32], 3: [128, 64, 32]}
WD_GRID = [1e-5, 1e-4, 1e-3]         # Adam weight decay (L2), chosen per month on the validation block
LR = 1e-3                             # initial learning rate; halved when validation stalls for 2 epochs
BATCH = 10_000
PATIENCE = 5                          # early stopping: epochs without improvement of validation rank correlation
VALID_FRAC = 0.2                      # last 20% of the window's dates validate
WINDOW = 252
TRAIN_END_DATE = '2011-12-31'

FEATURE_SETS = {"core": "core", "all": "all", "embed+norm+core": "textcore", "embed+norm+all": "textall",
                "embed": "textonly", "embed+norm": "textonly"}
CORE_COLS = ['net_sentiment', 'log_volume']
TARGET_TAGS = {"f_cumret1": "", "ar_dgtw_1": "_dgtw"}
assert NN_ARCH in ARCHS and FEATURE_SET in FEATURE_SETS and TARGET_COL in TARGET_TAGS

MODEL_NAME = (f"nn{NN_ARCH}_{FEATURE_SETS[FEATURE_SET]}" + ("_rank" if RANK_TARGET else "") + TARGET_TAGS[TARGET_COL]
              + ("" if RANK_FEATURES else "_norank") + ("_placebo" if PLACEBO else ""))
print(f"NN_ARCH={NN_ARCH} {ARCHS[NN_ARCH]}  FEATURE_SET={FEATURE_SET}  RANK_TARGET={RANK_TARGET}  TARGET_COL={TARGET_COL}  "
      f"RANK_FEATURES={RANK_FEATURES}  PLACEBO={PLACEBO}  N_SEEDS={N_SEEDS}  WD_GRID={WD_GRID}")
print(f"workers={N_WORKERS} x threads={THREADS_PER_WORKER}; max epochs {MAX_EPOCHS}; -> model name {MODEL_NAME}")

## Data

In [ ]:
# Load data (row label = merged_master row label, so predictions can be placed back onto the panel)
t0 = time.time()
data = pd.read_pickle(INPUT_DATA).set_index("mm_index")
data["date"] = pd.to_datetime(data["date"])
print(f"Loaded {len(data):,} stock-days with text, {data['date'].min().date()} to {data['date'].max().date()} [{time.time() - t0:.0f}s]")

In [ ]:
# Features and target (same definitions as the linear text notebook)
TARGET = TARGET_COL
EMBED_COLS = [c for c in data.columns if c.startswith('embed_') and c not in ('embed_n', 'embed_norm', 'embed_cos')]
NORM_COLS = ['embed_norm', 'embed_cos']
assert len(EMBED_COLS) == 384, len(EMBED_COLS)
ALL_COLS = [c for c in data.columns
            if not (c.startswith('ar_') or c.startswith('embed_') or c.startswith('px_')
                    or c in ('permno', 'ticker', 'date', 'f_cumret1', 'year_month'))]
assert len(ALL_COLS) == 53, (len(ALL_COLS), ALL_COLS)
FEATURES = {"embed": EMBED_COLS,
            "embed+norm": EMBED_COLS + NORM_COLS,
            "core": CORE_COLS,
            "embed+norm+core": EMBED_COLS + NORM_COLS + CORE_COLS,
            "all": ALL_COLS,
            "embed+norm+all": EMBED_COLS + NORM_COLS + ALL_COLS}[FEATURE_SET]

# Prediction coverage and feature ranks depend only on available inputs.
# A missing future return must not change today's feature cross-section.
model_data = data[[TARGET] + FEATURES].dropna(subset=FEATURES)
model_data[TARGET] = model_data[TARGET].astype('float64')
model_data['date'] = data.loc[model_data.index, 'date']
keys = data[['permno', 'ticker']]
del data
model_data = model_data.sort_values('date', kind='stable')
if RANK_TARGET:
    model_data[TARGET] = model_data.groupby('date')[TARGET].rank(pct=True) - 0.5
    print(f"RANK_TARGET: {TARGET} replaced by its daily percentile rank - 0.5")

# GKX: non-embedding features become daily percentile ranks mapped to [-1, 1]; embeddings stay as they are
NONEMBED = [c for c in FEATURES if not c.startswith('embed_')]
if RANK_FEATURES and NONEMBED:
    g = model_data.groupby('date')
    for c in NONEMBED:
        model_data[c] = (g[c].rank(pct=True) * 2 - 1).astype('float32')
    print(f"RANK_FEATURES: {len(NONEMBED)} non-embedding features replaced by daily ranks in [-1, 1]")
if PLACEBO:
    rng = np.random.default_rng(0)
    for k in range(PLACEBO):
        model_data[f'placebo_{k}'] = rng.standard_normal(len(model_data)).astype('float32')
    FEATURES = FEATURES + [f'placebo_{k}' for k in range(PLACEBO)]
    print(f"PLACEBO: {PLACEBO} standard-normal noise columns appended")

# Feature groups for the importance diagnostic
GROUPS = {}
if any(c.startswith('embed_') and c not in NORM_COLS for c in FEATURES):
    GROUPS['embedding_384'] = [i for i, c in enumerate(FEATURES) if c in EMBED_COLS]
if any(c in NORM_COLS for c in FEATURES):
    GROUPS['agreement_2'] = [i for i, c in enumerate(FEATURES) if c in NORM_COLS]
if any(c in CORE_COLS for c in FEATURES):
    GROUPS['core_2'] = [i for i, c in enumerate(FEATURES) if c in CORE_COLS]
other = [i for i, c in enumerate(FEATURES) if c in ALL_COLS and c not in CORE_COLS]
if other:
    GROUPS['other_51'] = other
if PLACEBO:
    GROUPS['placebo'] = [i for i, c in enumerate(FEATURES) if c.startswith('placebo_')]
CORE_IDX = {c: FEATURES.index(c) for c in CORE_COLS if c in FEATURES}

print(f"Sample size: {len(model_data):,}   target: {TARGET}   features: {len(FEATURES)}   groups: {list(GROUPS)}")

In [ ]:
# A configuration-specific directory isolates pilots, full runs and concurrent jobs.
# Identity includes the source data stat, feature names, code and numerical-library versions.
stat = INPUT_DATA.stat()
notebook_path = CODE_DIR / "03d - neural network" / "prediction_neural_network_text.ipynb"
RUN_SPEC = {
    "model": MODEL_NAME, "kind": RUN_KIND, "features": FEATURES, "arch": ARCHS[NN_ARCH],
    "target": TARGET_COL, "rank_target": RANK_TARGET, "rank_features": RANK_FEATURES,
    "placebo": PLACEBO, "window": WINDOW, "train_end": TRAIN_END_DATE,
    "wd_grid": WD_GRID, "lr": LR, "batch": BATCH, "patience": PATIENCE, "valid_frac": VALID_FRAC,
    "n_seeds": N_SEEDS, "max_epochs": MAX_EPOCHS, "max_months": MAX_MONTHS, "months": RUN_MONTHS,
    "threads": THREADS_PER_WORKER, "seed_formula": "torch=1000*seed+7; batch_shuffle=seed",
    "input": {"path": str(INPUT_DATA), "bytes": stat.st_size, "mtime_ns": stat.st_mtime_ns},
    "code_sha256": {str(path.relative_to(CODE_DIR)): hashlib.sha256(path.read_bytes()).hexdigest()
                    for path in [notebook_path, CODE_DIR / "tools" / "nn_checkpoint.py", CODE_DIR / "tools" / "prediction_metrics.py"]},
    "versions": {name: importlib.metadata.version(name)
                 for name in ["numpy", "pandas", "torch", "joblib", "filelock"]}}
RUN_FINGERPRINT = fingerprint(RUN_SPEC)
RUN_DIR = RUN_ROOT / f"{MODEL_NAME}_{RUN_KIND}_{RUN_FINGERPRINT[:12]}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR = RUN_DIR / "arrays"
TMP_DIR.mkdir(exist_ok=True)
X_PATH = TMP_DIR / "X.npy"
n, p = len(model_data), len(FEATURES)
t0 = time.time()
with FileLock(str(RUN_DIR / "prepare.lock"), timeout=0):
    manifest = RUN_DIR / "run_spec.json"
    if manifest.exists() and json.loads(manifest.read_text()) != RUN_SPEC:
        raise ValueError("Run identity mismatch; existing files were preserved")
    atomic_json(RUN_SPEC, manifest)
    if X_PATH.exists():
        check = np.load(X_PATH, mmap_mode="r")
        assert check.shape == (n, p) and check.dtype == np.float32
        del check
        print("Reusing verified design matrix")
    else:
        temporary = TMP_DIR / f"X_{os.getpid()}.tmp.npy"
        try:
            X_mm = np.lib.format.open_memmap(temporary, mode="w+", dtype=np.float32, shape=(n, p))
            for start in range(0, n, 200_000):
                block = model_data.iloc[start:start+200_000][FEATURES].to_numpy(dtype=np.float32)
                if not np.isfinite(block).all():
                    raise ValueError("Nonfinite design matrix")
                X_mm[start:start+200_000] = block
            X_mm.flush()
            del X_mm, block
            os.replace(temporary, X_PATH)
        finally:
            temporary.unlink(missing_ok=True)
y_all = model_data[TARGET].to_numpy(dtype=np.float32)
if np.isinf(y_all).any():
    raise ValueError("Infinite training target")
print(f"Missing labels retained for prediction coverage: {np.isnan(y_all).sum():,}")
dates_all = model_data["date"].to_numpy().astype("datetime64[ns]")
index_all = model_data.index.to_numpy()
date_codes_all = pd.factorize(model_data["date"])[0].astype(np.int32)
del model_data
print(f"Run: {RUN_DIR}")
print(f"Design matrix {n:,} x {p} ({X_PATH.stat().st_size / 1024**3:.1f} GB) [{time.time()-t0:.0f}s]")


## OOS predictions (monthly training, daily predictions)

In [ ]:
# Dates, months and the row ranges of every month's window / validation block / test month
unique_dates = pd.DatetimeIndex(np.unique(dates_all))
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]
oos_months = sorted(pd.Series(oos_dates).dt.to_period('M').unique())
if RUN_MONTHS:
    requested = [pd.Period(m, freq="M") for m in RUN_MONTHS]
    missing = set(requested)-set(oos_months)
    if missing:
        raise ValueError(f"Requested months are absent: {missing}")
    oos_months = [m for m in oos_months if m in requested]
if MAX_MONTHS:
    oos_months = oos_months[:MAX_MONTHS]
    print(f"MAX_MONTHS={MAX_MONTHS}: smoke test on {[str(m) for m in oos_months]}")


def month_window(pred_month):
    # the 252 trading days before pred_month, and the month's own prediction dates
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    if len(month_dates) == 0:
        return None
    train_cutoff = pred_month.to_timestamp() - pd.Timedelta(days=2)
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return None
    last_idx = unique_dates.get_loc(train_dates[-1])
    if last_idx < WINDOW:
        return None
    return unique_dates[last_idx - WINDOW + 1], train_dates[-1], month_dates


def month_task(pred_month):
    w = month_window(pred_month)
    if w is None:
        return None
    start_date, last_train_date, month_dates = w
    window_dates = unique_dates[(unique_dates >= start_date) & (unique_dates <= last_train_date)]
    valid_start = window_dates[int(round(len(window_dates) * (1 - VALID_FRAC)))]
    d64 = dates_all
    lo = int(np.searchsorted(d64, np.datetime64(start_date, 'ns'), 'left'))
    vs = int(np.searchsorted(d64, np.datetime64(valid_start, 'ns'), 'left'))
    hi = int(np.searchsorted(d64, np.datetime64(last_train_date, 'ns'), 'right'))
    t0_ = int(np.searchsorted(d64, np.datetime64(month_dates[0], 'ns'), 'left'))
    t1_ = int(np.searchsorted(d64, np.datetime64(month_dates[-1], 'ns'), 'right'))
    if hi <= vs or vs <= lo or t1_ <= t0_:
        return None
    return {'month': str(pred_month), 'lo': lo, 'vs': vs, 'hi': hi, 't0': t0_, 't1': t1_}


tasks = [t for t in (month_task(m) for m in oos_months) if t is not None]
if not tasks:
    raise ValueError("No months with a full training/validation window")
for task in tasks:
    assert task["lo"] < task["vs"] < task["hi"] <= task["t0"] < task["t1"]
print(f"Rolling window: {WINDOW} trading days, validation = last {VALID_FRAC:.0%} of its dates")
print(f"OOS months with a full window: {len(tasks)} ({tasks[0]['month']} to {tasks[-1]['month']})")
print(f"First month: train rows {tasks[0]['vs'] - tasks[0]['lo']:,}, validation rows {tasks[0]['hi'] - tasks[0]['vs']:,}, "
      f"test rows {tasks[0]['t1'] - tasks[0]['t0']:,}")

In [ ]:
def daily_rank_corr(p, y, codes):
    from prediction_metrics import rank_correlation
    table = pd.DataFrame({"date": codes, "prediction": p, "target": y})
    rc = rank_correlation(table, "prediction", "target")
    return float(rc.mean()) if rc.notna().any() else np.nan


CONFIG = {'arch': ARCHS[NN_ARCH], 'wd_grid': WD_GRID, 'lr': LR, 'batch': BATCH, 'patience': PATIENCE,
          'max_epochs': MAX_EPOCHS, 'n_seeds': N_SEEDS, 'threads': THREADS_PER_WORKER, 'groups': GROUPS,
          'core_idx': CORE_IDX, 'x_path': str(X_PATH), 'p': p, 'rank_features': RANK_FEATURES,
          'checkpoint_dir': str(RUN_DIR / 'months'), 'fingerprint': RUN_FINGERPRINT}


def fit_month(task, y_all, codes_all, cfg):
    # Runs in a worker process: trains every (weight decay, seed) network for one month, chooses the weight
    # decay on the validation block, returns the seed-averaged test predictions and the diagnostics.
    import numpy as np
    import torch
    torch.set_num_threads(cfg['threads'])
    X = np.load(cfg['x_path'], mmap_mode='r')
    lo, vs, hi, t0_, t1_ = task['lo'], task['vs'], task['hi'], task['t0'], task['t1']
    Xtr = np.array(X[lo:vs]); Xva = np.array(X[vs:hi]); Xte = np.array(X[t0_:t1_])
    ytr, yva, yte = y_all[lo:vs], y_all[vs:hi], y_all[t0_:t1_]
    known = np.isfinite(ytr)
    Xtr, ytr = Xtr[known], ytr[known]
    if len(ytr) < 10 or np.isfinite(yva).sum() < 10:
        raise ValueError(f"Insufficient labeled training/validation rows for {task['month']}")
    cva, cte = codes_all[vs:hi], codes_all[t0_:t1_]
    mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
    sd[sd < 1e-8] = 1.0
    for A in (Xtr, Xva, Xte):
        A -= mu; A /= sd
    ybar = float(ytr.mean())
    Xtr_t, ytr_t = torch.from_numpy(Xtr), torch.from_numpy(ytr - ybar)
    Xva_t, Xte_t = torch.from_numpy(Xva), torch.from_numpy(Xte)
    n_tr, p_ = Xtr.shape

    def make_model():
        layers, d = [], p_
        for h in cfg['arch']:
            layers += [torch.nn.Linear(d, h), torch.nn.BatchNorm1d(h), torch.nn.ReLU()]
            d = h
        layers.append(torch.nn.Linear(d, 1))
        return torch.nn.Sequential(*layers)

    def predict(model, Xt):
        model.eval()
        with torch.no_grad():
            out = []
            for s in range(0, len(Xt), 200_000):
                out.append(model(Xt[s:s + 200_000]).squeeze(1))
            return torch.cat(out).numpy() + ybar

    res = {}     # wd -> dict(models, val_rc per seed, epochs per seed)
    for wd in cfg['wd_grid']:
        models, val_rcs, epochs = [], [], []
        for seed in range(cfg['n_seeds']):
            torch.manual_seed(1000 * seed + 7)
            gen = torch.Generator().manual_seed(seed)
            model = make_model()
            opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=wd)
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
            best_rc, best_state, best_epoch, bad = -np.inf, None, 0, 0
            for epoch in range(1, cfg['max_epochs'] + 1):
                model.train()
                perm = torch.randperm(n_tr, generator=gen)
                for s in range(0, n_tr, cfg['batch']):
                    idx = perm[s:s + cfg['batch']]
                    if len(idx) < 2:
                        continue
                    opt.zero_grad()
                    loss = torch.nn.functional.mse_loss(model(Xtr_t[idx]).squeeze(1), ytr_t[idx])
                    loss.backward()
                    opt.step()
                rc = daily_rank_corr(predict(model, Xva_t), yva, cva)
                if not np.isfinite(rc):
                    raise ValueError("Validation has no eligible nonconstant target cross-sections")
                sched.step(rc)
                if rc > best_rc + 1e-5:
                    best_rc, best_epoch, bad = rc, epoch, 0
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                else:
                    bad += 1
                    if bad >= cfg['patience']:
                        break
            model.load_state_dict(best_state)
            models.append(model); val_rcs.append(best_rc); epochs.append(best_epoch)
        res[wd] = {'models': models, 'val_rc': val_rcs, 'epochs': epochs}

    # ensemble predictions per weight decay; realised OOS rank correlation of every candidate (for the sidecar)
    preds_te, preds_va, test_rc, val_rc_ens = {}, {}, {}, {}
    for wd, r in res.items():
        preds_te[wd] = np.mean([predict(m, Xte_t) for m in r['models']], axis=0)
        preds_va[wd] = np.mean([predict(m, Xva_t) for m in r['models']], axis=0)
        score = daily_rank_corr(preds_te[wd], yte, cte)
        test_rc[wd] = score if np.isfinite(score) else None
        val_rc_ens[wd] = daily_rank_corr(preds_va[wd], yva, cva)
    chosen = max(cfg['wd_grid'], key=lambda w: float(np.mean(res[w]['val_rc'])))
    ens = res[chosen]['models']

    # group importance on the validation block: set the group to its training mean (= 0 after standardisation)
    importance = {}
    base = val_rc_ens[chosen]
    for gname, cols in cfg['groups'].items():
        Xz = Xva.copy(); Xz[:, cols] = 0.0
        pz = np.mean([predict(m, torch.from_numpy(Xz)) for m in ens], axis=0)
        importance[gname] = base - daily_rank_corr(pz, yva, cva)

    # marginal response of the prediction to each core feature (others at their median) and their interaction
    marginal, marginal_grids, interaction = {}, {}, None
    if cfg['core_idx']:
        med = np.median(Xva, axis=0).astype(np.float32)
        grid = np.linspace(-1.0, 1.0, 21)
        for cname, j in cfg['core_idx'].items():
            feature_grid = grid if cfg['rank_features'] else np.quantile(Xtr[:, j] * sd[j] + mu[j], np.linspace(.01, .99, 21))
            marginal_grids[cname] = feature_grid.tolist()
            Xg = np.tile(med, (len(feature_grid), 1)); Xg[:, j] = (feature_grid - mu[j]) / sd[j]
            marginal[cname] = np.mean([predict(m, torch.from_numpy(Xg)) for m in ens], axis=0).tolist()
        if len(cfg['core_idx']) == 2 and cfg['rank_features']:
            (c1, j1), (c2, j2) = cfg['core_idx'].items()
            g11 = np.linspace(-1.0, 1.0, 11)
            Xg = np.tile(med, (len(g11) ** 2, 1))
            a, b = np.meshgrid(g11, g11, indexing='ij')
            Xg[:, j1] = (a.ravel() - mu[j1]) / sd[j1]; Xg[:, j2] = (b.ravel() - mu[j2]) / sd[j2]
            interaction = {'rows': c1, 'cols': c2, 'grid': g11.tolist(),
                           'pred': np.mean([predict(m, torch.from_numpy(Xg)) for m in ens], axis=0).reshape(11, 11).tolist()}

    return {'month': task['month'], 'rows': (t0_, t1_), 'pred': preds_te[chosen], 'chosen_wd': chosen,
            'val_rc_by_wd_seed': {str(w): r['val_rc'] for w, r in res.items()},
            'val_rc_ens_by_wd': {str(w): v for w, v in val_rc_ens.items()},
            'test_rc_by_wd': {str(w): v for w, v in test_rc.items()},
            'epochs_by_wd_seed': {str(w): r['epochs'] for w, r in res.items()},
            'importance': importance, 'marginal': marginal, 'marginal_grids': marginal_grids, 'interaction': interaction,
            'n_train': int(n_tr), 'n_valid': int(np.isfinite(yva).sum()), 'n_test': int(t1_ - t0_)}

In [ ]:
# Each finished month is atomically checkpointed by the worker. OS-backed locks
# release automatically if a process dies. A rerun verifies and reuses completed months.
t0 = time.time()
print(f"Fitting/resuming {len(tasks)} months x {len(WD_GRID)} penalties x {N_SEEDS} seeds; "
      f"{N_WORKERS} workers x {THREADS_PER_WORKER} threads")
with FileLock(str(RUN_DIR / "training.lock"), timeout=0):
    results = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=5, max_nbytes="1M",
                       temp_folder=str(TMP_DIR))(
        delayed(fit_month_checkpoint)(task, y_all, date_codes_all, CONFIG, fit_month) for task in tasks)
results = sorted(results, key=lambda r: r["month"])
assert len(results) == len(tasks)
print(f"Completed {len(results)} months in {(time.time()-t0)/60:.1f} minutes")


In [ ]:
# Summaries: chosen weight decay, epochs, validation vs realised rank correlation of every candidate
chosen = pd.Series({r['month']: r['chosen_wd'] for r in results}, name='weight_decay')
print("Chosen weight decay by month (validation block):")
print(chosen.value_counts().sort_index().rename('months').to_string())
val_tbl = pd.DataFrame({r['month']: r['val_rc_ens_by_wd'] for r in results}).T.sort_index()
test_tbl = pd.DataFrame({r['month']: r['test_rc_by_wd'] for r in results}).T.sort_index()
ep = pd.DataFrame({r['month']: {w: np.mean(e) for w, e in r['epochs_by_wd_seed'].items()} for r in results}).T.sort_index()
print("\nMean daily rank correlation by candidate weight decay (whole OOS period, for information only):")
print(pd.DataFrame({'validation (seed ensemble)': val_tbl.mean(), 'realised OOS': test_tbl.mean(),
                    'mean best epoch': ep.mean()}).round(4).to_string())
realised_chosen = pd.Series({r['month']: r['test_rc_by_wd'][str(r['chosen_wd'])] for r in results}).sort_index()
print(f"\nRealised OOS rank correlation at the chosen setting: mean {realised_chosen.mean():.4f} "
      f"(months {len(realised_chosen)}; {(realised_chosen > 0).mean():.0%} positive)")
seed_spread = pd.DataFrame({r['month']: {'sd_across_seeds': np.std(r['val_rc_by_wd_seed'][str(r['chosen_wd'])])} for r in results}).T
print(f"Validation rank correlation: sd across seeds at the chosen setting, median over months {seed_spread['sd_across_seeds'].median():.4f}")

In [ ]:
# Diagnostics: group importance (validation block), placebo, marginal responses
imp = pd.DataFrame({r['month']: r['importance'] for r in results}).T.sort_index()
imp_mean = imp.mean()
print("Group importance = drop in validation rank correlation when the group is set to its mean (average over months):")
print(pd.DataFrame({'drop': imp_mean.round(5), 'share': (imp_mean.clip(lower=0) / max(imp_mean.clip(lower=0).sum(), 1e-12)).round(3),
                    'months_positive': (imp > 0).mean().round(2)}).to_string())
marg = {}
for cname in CORE_IDX:
    curves = np.array([r['marginal'][cname] for r in results if r['marginal']])
    marg[cname] = curves.mean(axis=0).tolist()
    print(f"\nMarginal response to {cname} (daily rank in [-1, 1], 21 points, mean over months; prediction units = target units):")
    print(np.round(marg[cname], 4))
inter = None
if results[0]['interaction'] is not None:
    inter = {'rows': results[0]['interaction']['rows'], 'cols': results[0]['interaction']['cols'],
             'grid': results[0]['interaction']['grid'],
             'pred': np.mean([r['interaction']['pred'] for r in results], axis=0).tolist()}
    print(f"\nInteraction surface: rows = {inter['rows']}, cols = {inter['cols']} (11 x 11 grid on [-1, 1], mean over months):")
    print(pd.DataFrame(np.round(inter['pred'], 4), index=np.round(inter['grid'], 1), columns=np.round(inter['grid'], 1)).to_string())

In [ ]:
# Assemble the predictions (chosen setting per month)
rows = []
for r in results:
    a, b = r['rows']
    rows.append(pd.DataFrame({'date': dates_all[a:b], 'index': index_all[a:b], 'prediction': r['pred'].astype(np.float64)}))
predictions_df = pd.concat(rows, ignore_index=True)
predictions_df = predictions_df.merge(keys.reset_index().rename(columns={'mm_index': 'index'}), on='index', how='left')
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']].sort_values(['date', 'ticker']).reset_index(drop=True)
print(f"Total predictions: {len(predictions_df):,}   non-null: {predictions_df['prediction'].notna().sum():,}")
print(f"Prediction spread: std {predictions_df['prediction'].std():.5f}, "
      f"1st/99th pct {predictions_df['prediction'].quantile(0.01):.5f} / {predictions_df['prediction'].quantile(0.99):.5f}")
print(predictions_df.head(10))
assert not predictions_df.duplicated(["date", "permno"]).any()
assert predictions_df["index"].is_unique
assert np.isfinite(predictions_df["prediction"]).all()
expected_dates = pd.DatetimeIndex(np.concatenate([dates_all[t["t0"]:t["t1"]] for t in tasks])).unique()
assert set(predictions_df["date"]) == set(expected_dates)


In [ ]:
# Atomic publication within this run only. Existing production predictions are untouched.
OUTPUT_FILE = RUN_DIR / f"predictions_{MODEL_NAME}_input={len(FEATURES)}.pkl"
atomic_pickle(predictions_df, OUTPUT_FILE)
side = {'fingerprint': RUN_FINGERPRINT, 'run_kind': RUN_KIND, 'run_spec': RUN_SPEC, 'feature_columns': FEATURES, 'model': MODEL_NAME, 'arch': ARCHS[NN_ARCH], 'feature_set': FEATURE_SET, 'features': len(FEATURES),
        'target': TARGET_COL, 'rank_target': RANK_TARGET, 'rank_features': RANK_FEATURES, 'placebo': PLACEBO,
        'window': WINDOW, 'valid_frac': VALID_FRAC, 'train_end': TRAIN_END_DATE, 'wd_grid': WD_GRID, 'lr': LR,
        'batch': BATCH, 'patience': PATIENCE, 'max_epochs': MAX_EPOCHS, 'n_seeds': N_SEEDS, 'max_months': MAX_MONTHS,
        'selection': 'weight decay with the best mean validation rank correlation across seeds (last 20% of the window)',
        'chosen_wd_by_month': {k: v for k, v in chosen.items()},
        'val_rc_ens_by_month_and_wd': {r['month']: r['val_rc_ens_by_wd'] for r in results},
        'val_rc_by_month_wd_seed': {r['month']: r['val_rc_by_wd_seed'] for r in results},
        'test_rc_by_month_and_wd': {r['month']: r['test_rc_by_wd'] for r in results},
        'epochs_by_month_wd_seed': {r['month']: r['epochs_by_wd_seed'] for r in results},
        'n_by_month': {r['month']: [r['n_train'], r['n_valid'], r['n_test']] for r in results},
        'group_importance_by_month': {r['month']: r['importance'] for r in results},
        'group_importance_mean': imp_mean.to_dict(),
        'marginal_mean': marg, 'interaction_mean': inter,
        'created': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}
side["marginal_grids_by_month"] = {r["month"]: r["marginal_grids"] for r in results}
side["elapsed_seconds_by_month"] = {r["month"]: r["elapsed_seconds"] for r in results}
atomic_json(side, OUTPUT_FILE.with_suffix(".json"))
atomic_json({"fingerprint": RUN_FINGERPRINT, "months": [r["month"] for r in results],
             "rows": len(predictions_df), "prediction_file": str(OUTPUT_FILE),
             "status": "complete", "kind": RUN_KIND}, RUN_DIR / "complete.json")
print(f"Predictions: {OUTPUT_FILE}")
print(f"Sidecar: {OUTPUT_FILE.with_suffix('.json')}")
print("Arrays/checkpoints retained for fast resume; outputs are isolated by run fingerprint.")
